In [16]:
from dotenv import load_dotenv
load_dotenv()

True

In [41]:
### context_schema

In [17]:
### agent will not directly use the context you provide, you need to access it using tool or prompt

In [18]:
from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model='gemini-3.1-flash-lite-preview')

In [35]:
from dataclasses import dataclass
@dataclass
class UserContext: 
    name: str
    age : int

In [20]:
from langchain.agents import create_agent

agent = create_agent(model = model, 
                     context_schema = {
                         "name" : "Nitin", 
                         "age" : 24
                     },
                     system_prompt = 'You are an helpful assistant')

In [21]:
from langchain.messages import HumanMessage
response = agent.invoke({"messages" : [HumanMessage('What is my name?')]})
response

{'messages': [HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}, id='e8658a1f-9d6f-429a-9a32-1f3afc86aa64'),
  AIMessage(content=[{'type': 'text', 'text': "I don't know your name. As an AI, I don't have access to your personal information or identity unless you have shared it with me in our current conversation. \n\nIf you'd like, you can tell me your name, and I'll be happy to remember it for the rest of our chat!", 'extras': {'signature': 'EjQKMgEMOdbHcF6npxZRgQ6h7wH4/u/mqw68bGWMS6D81Jv+XnTg05kPzSnIwKKlHFmAr41L'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019db887-0516-7301-878a-8cebfba109af-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 67, 'total_tokens': 78, 'input_token_details': {'cache_read': 0}})]}

In [22]:
### using tool accessing context schema

In [31]:
from langchain.tools import tool, ToolRuntime
@tool
def get_name(context: UserContext)-> str:
    '''function to get user name from UserContext'''
    print('context for get_name', context)
    return context['name']

@tool
def get_age(context: UserContext)-> int:
    '''function to get user age from UserContext'''
    return context['age']

agent = create_agent(model = model, 
                     tools = [get_name, get_age],
                     context_schema = UserContext, 
                     system_prompt = 'You are an helpful assistant')

In [32]:
from langchain.messages import HumanMessage
response = agent.invoke({"messages" : [HumanMessage('What is my name?')]}, context = UserContext(name='Nitin', age=24))
response

context for get_name {'name': 'John Doe', 'age': 30}


{'messages': [HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}, id='b4d86798-cf6a-4a43-9a9a-f6c3a276d257'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_name', 'arguments': '{"context": {"name": "John Doe", "age": 30}}'}, '__gemini_function_call_thought_signatures__': {'694e2ea1-8c11-4e72-9f2d-9065fc93cf2e': 'EjQKMgEMOdbHGAYadTxBx+7OV0ekXRPwaBXvVlkY/B7He0B4RR9srHyy+Kc1buKaQV5qFDU+'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019db890-1163-7e11-b062-6fa51a73dd2b-0', tool_calls=[{'name': 'get_name', 'args': {'context': {'name': 'John Doe', 'age': 30}}, 'id': '694e2ea1-8c11-4e72-9f2d-9065fc93cf2e', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 157, 'output_tokens': 24, 'total_tokens': 181, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content='John Doe', name='get_n

In [38]:
from langchain.tools import tool, ToolRuntime
@tool
def get_name(runtime: ToolRuntime)-> str:
    '''function to get user name '''
    print('runtime for get_name', runtime)
    return runtime.context.name

@tool
def get_age(runtime: ToolRuntime)-> int:
    '''function to get user age '''
    return runtime.context.age

agent = create_agent(model = model, 
                     tools = [get_name, get_age],
                     context_schema = UserContext, 
                     system_prompt = 'You are an helpful assistant')

In [39]:
from langchain.messages import HumanMessage
response = agent.invoke({"messages" : [HumanMessage('What is my name?')]}, context = UserContext(name='Nitin', age=24))


/opt/homebrew/Caskroom/miniconda/base/envs/lca-lc-foundation/lib/python3.13/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=UserContext(name='Nitin', age=24), input_type=UserContext])
  return self.__pydantic_serializer__.to_python(


runtime for get_name ToolRuntime(state={'messages': [HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}, id='5b2250e3-657a-4557-8069-b8a6c84354df'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_name', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'5be7c049-1d8a-4919-918a-19f887a7eeb6': 'EjQKMgEMOdbHcT/DkeFiKGod5zEpIlZK3CM2OzIxS/njjjI04vBNjLEnHqjiUGr7qdaju+B/'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019db893-57a2-74e1-9b93-b8dbe287cbe1-0', tool_calls=[{'name': 'get_name', 'args': {}, 'id': '5be7c049-1d8a-4919-918a-19f887a7eeb6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 47, 'output_tokens': 10, 'total_tokens': 57, 'input_token_details': {'cache_read': 0}})]}, context=UserContext(name='Nitin', age=24), config={'tags': [], 'metadata': {'ls_integration'

In [40]:
response

{'messages': [HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}, id='5b2250e3-657a-4557-8069-b8a6c84354df'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_name', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'5be7c049-1d8a-4919-918a-19f887a7eeb6': 'EjQKMgEMOdbHcT/DkeFiKGod5zEpIlZK3CM2OzIxS/njjjI04vBNjLEnHqjiUGr7qdaju+B/'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019db893-57a2-74e1-9b93-b8dbe287cbe1-0', tool_calls=[{'name': 'get_name', 'args': {}, 'id': '5be7c049-1d8a-4919-918a-19f887a7eeb6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 47, 'output_tokens': 10, 'total_tokens': 57, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content='Nitin', name='get_name', id='6e3077c3-601b-403f-9251-0ff9714a1bc7', tool_call_id='5be7c049-1d8a-4919-918a-19

In [42]:
### state_schema:

In [59]:
from langchain.agents import AgentState

class StateUserContext(AgentState):
    age : int


In [74]:
from langchain.messages import ToolMessage
from langgraph.types import Command
@tool
def update_age(age: int, runtime: ToolRuntime)-> Command:
    """funtion to update user age and save in context"""
    print(f'update age called : {runtime.state,age}')
    return Command (
        update={
            "age" : age, 
            # update the message list as well
            "messages" : [ToolMessage("Successfully updated the user age", tool_call_id = runtime.tool_call_id)]
        }
    )
@tool
def get_age(runtime: ToolRuntime)-> int:
    '''function to get user age '''
    print(f'get age called : {runtime.state}')
    return runtime.state['age']

In [75]:
from langgraph.checkpoint.memory import InMemorySaver
agent = create_agent(model = model , 
                    tools = [update_age, get_age, get_name], 
                    state_schema = StateUserContext, 
                     checkpointer = InMemorySaver(),
                    system_prompt = 'You are an helpful assistant')
config = {"configurable" : {"thread_id" : 1}}

In [76]:
response = agent.invoke({"messages": [HumanMessage("My age is 69")]}, config = config)
response

/opt/homebrew/Caskroom/miniconda/base/envs/lca-lc-foundation/lib/python3.13/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='thread_id', input_value=1, input_type=int])
  return self.__pydantic_serializer__.to_python(


update age called : ({'messages': [HumanMessage(content='My age is 69', additional_kwargs={}, response_metadata={}, id='f06e497b-ea9b-401a-b778-900372d580b0'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'update_age', 'arguments': '{"age": 69}'}, '__gemini_function_call_thought_signatures__': {'b59306c3-8b40-4193-9439-26c0ea5108ce': 'EjQKMgEMOdbHHjCGRRtQuSuGRXiluTxL80KvGKr6GsxpfPX/q7VyDBu/bVNvxngPHV/ED/nc'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019db8ba-f7af-7fd0-a8d9-35d1aafc334c-0', tool_calls=[{'name': 'update_age', 'args': {'age': 69}, 'id': 'b59306c3-8b40-4193-9439-26c0ea5108ce', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 96, 'output_tokens': 15, 'total_tokens': 111, 'input_token_details': {'cache_read': 0}})]}, 69)


{'messages': [HumanMessage(content='My age is 69', additional_kwargs={}, response_metadata={}, id='f06e497b-ea9b-401a-b778-900372d580b0'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'update_age', 'arguments': '{"age": 69}'}, '__gemini_function_call_thought_signatures__': {'b59306c3-8b40-4193-9439-26c0ea5108ce': 'EjQKMgEMOdbHHjCGRRtQuSuGRXiluTxL80KvGKr6GsxpfPX/q7VyDBu/bVNvxngPHV/ED/nc'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019db8ba-f7af-7fd0-a8d9-35d1aafc334c-0', tool_calls=[{'name': 'update_age', 'args': {'age': 69}, 'id': 'b59306c3-8b40-4193-9439-26c0ea5108ce', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 96, 'output_tokens': 15, 'total_tokens': 111, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content='Successfully updated the user age', name='update_age', id='5ae8c5e8-7c65-44ac-b0a8-c2cc5c

In [77]:
response = agent.invoke({"messages": [HumanMessage("what is my age?")]},  config = config)
response

/opt/homebrew/Caskroom/miniconda/base/envs/lca-lc-foundation/lib/python3.13/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='thread_id', input_value=1, input_type=int])
  return self.__pydantic_serializer__.to_python(


get age called : {'messages': [HumanMessage(content='My age is 69', additional_kwargs={}, response_metadata={}, id='f06e497b-ea9b-401a-b778-900372d580b0'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'update_age', 'arguments': '{"age": 69}'}, '__gemini_function_call_thought_signatures__': {'b59306c3-8b40-4193-9439-26c0ea5108ce': 'EjQKMgEMOdbHHjCGRRtQuSuGRXiluTxL80KvGKr6GsxpfPX/q7VyDBu/bVNvxngPHV/ED/nc'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019db8ba-f7af-7fd0-a8d9-35d1aafc334c-0', tool_calls=[{'name': 'update_age', 'args': {'age': 69}, 'id': 'b59306c3-8b40-4193-9439-26c0ea5108ce', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 96, 'output_tokens': 15, 'total_tokens': 111, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='Successfully updated the user age', name='update_age', id='5ae8c5e8-7c65-44a

{'messages': [HumanMessage(content='My age is 69', additional_kwargs={}, response_metadata={}, id='f06e497b-ea9b-401a-b778-900372d580b0'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'update_age', 'arguments': '{"age": 69}'}, '__gemini_function_call_thought_signatures__': {'b59306c3-8b40-4193-9439-26c0ea5108ce': 'EjQKMgEMOdbHHjCGRRtQuSuGRXiluTxL80KvGKr6GsxpfPX/q7VyDBu/bVNvxngPHV/ED/nc'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019db8ba-f7af-7fd0-a8d9-35d1aafc334c-0', tool_calls=[{'name': 'update_age', 'args': {'age': 69}, 'id': 'b59306c3-8b40-4193-9439-26c0ea5108ce', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 96, 'output_tokens': 15, 'total_tokens': 111, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content='Successfully updated the user age', name='update_age', id='5ae8c5e8-7c65-44ac-b0a8-c2cc5c

In [78]:
response = agent.invoke({"messages": [HumanMessage("My age is 32")]}, config = config)
response

/opt/homebrew/Caskroom/miniconda/base/envs/lca-lc-foundation/lib/python3.13/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='thread_id', input_value=1, input_type=int])
  return self.__pydantic_serializer__.to_python(


update age called : ({'messages': [HumanMessage(content='My age is 69', additional_kwargs={}, response_metadata={}, id='f06e497b-ea9b-401a-b778-900372d580b0'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'update_age', 'arguments': '{"age": 69}'}, '__gemini_function_call_thought_signatures__': {'b59306c3-8b40-4193-9439-26c0ea5108ce': 'EjQKMgEMOdbHHjCGRRtQuSuGRXiluTxL80KvGKr6GsxpfPX/q7VyDBu/bVNvxngPHV/ED/nc'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019db8ba-f7af-7fd0-a8d9-35d1aafc334c-0', tool_calls=[{'name': 'update_age', 'args': {'age': 69}, 'id': 'b59306c3-8b40-4193-9439-26c0ea5108ce', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 96, 'output_tokens': 15, 'total_tokens': 111, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='Successfully updated the user age', name='update_age', id='5ae8c5e8-7c65

{'messages': [HumanMessage(content='My age is 69', additional_kwargs={}, response_metadata={}, id='f06e497b-ea9b-401a-b778-900372d580b0'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'update_age', 'arguments': '{"age": 69}'}, '__gemini_function_call_thought_signatures__': {'b59306c3-8b40-4193-9439-26c0ea5108ce': 'EjQKMgEMOdbHHjCGRRtQuSuGRXiluTxL80KvGKr6GsxpfPX/q7VyDBu/bVNvxngPHV/ED/nc'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019db8ba-f7af-7fd0-a8d9-35d1aafc334c-0', tool_calls=[{'name': 'update_age', 'args': {'age': 69}, 'id': 'b59306c3-8b40-4193-9439-26c0ea5108ce', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 96, 'output_tokens': 15, 'total_tokens': 111, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content='Successfully updated the user age', name='update_age', id='5ae8c5e8-7c65-44ac-b0a8-c2cc5c

In [79]:
response = agent.invoke({"messages": [HumanMessage("what is my age?")]}, config = config)
response

/opt/homebrew/Caskroom/miniconda/base/envs/lca-lc-foundation/lib/python3.13/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='thread_id', input_value=1, input_type=int])
  return self.__pydantic_serializer__.to_python(


get age called : {'messages': [HumanMessage(content='My age is 69', additional_kwargs={}, response_metadata={}, id='f06e497b-ea9b-401a-b778-900372d580b0'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'update_age', 'arguments': '{"age": 69}'}, '__gemini_function_call_thought_signatures__': {'b59306c3-8b40-4193-9439-26c0ea5108ce': 'EjQKMgEMOdbHHjCGRRtQuSuGRXiluTxL80KvGKr6GsxpfPX/q7VyDBu/bVNvxngPHV/ED/nc'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019db8ba-f7af-7fd0-a8d9-35d1aafc334c-0', tool_calls=[{'name': 'update_age', 'args': {'age': 69}, 'id': 'b59306c3-8b40-4193-9439-26c0ea5108ce', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 96, 'output_tokens': 15, 'total_tokens': 111, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='Successfully updated the user age', name='update_age', id='5ae8c5e8-7c65-44a

{'messages': [HumanMessage(content='My age is 69', additional_kwargs={}, response_metadata={}, id='f06e497b-ea9b-401a-b778-900372d580b0'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'update_age', 'arguments': '{"age": 69}'}, '__gemini_function_call_thought_signatures__': {'b59306c3-8b40-4193-9439-26c0ea5108ce': 'EjQKMgEMOdbHHjCGRRtQuSuGRXiluTxL80KvGKr6GsxpfPX/q7VyDBu/bVNvxngPHV/ED/nc'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019db8ba-f7af-7fd0-a8d9-35d1aafc334c-0', tool_calls=[{'name': 'update_age', 'args': {'age': 69}, 'id': 'b59306c3-8b40-4193-9439-26c0ea5108ce', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 96, 'output_tokens': 15, 'total_tokens': 111, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content='Successfully updated the user age', name='update_age', id='5ae8c5e8-7c65-44ac-b0a8-c2cc5c